In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False


# ==========================================
# LOAD DATA
# ==========================================
df = pd.read_excel("Data_Model_IoTMLCQ_2024(11).xlsx")

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')


# ==========================================
# FEATURE ENGINEERING
# ==========================================

# interaksi fitur
df['temp_ph'] = (
    df['Temperature (°C)']
    * df['pH']
)

df['temp2'] = (
    df['Temperature (°C)'] ** 2
)

df['ph2'] = (
    df['pH'] ** 2
)

df['turb2'] = (
    df['Turbidity (NTU)'] ** 2
)

# lag feature
df['temp_prev'] = (
    df['Temperature (°C)']
    .shift(1)
)

df['ph_prev'] = (
    df['pH']
    .shift(1)
)

df['turb_prev'] = (
    df['Turbidity (NTU)']
    .shift(1)
)

df = df.dropna()


# ==========================================
# PILIH FITUR
# ==========================================
features = [
    'Temperature (°C)',
    'pH',
    'Turbidity (NTU)',

    'Month_Num',
    'day',
    'hour',

    'temp_ph',
    'temp2',
    'ph2',
    'turb2',

    'temp_prev',
    'ph_prev',
    'turb_prev'
]

X = df[features]
y = df['Dissolved Oxygen (mg/L)']


# ==========================================
# TRAIN TEST SPLIT
# ==========================================
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]


# ==========================================
# EVALUASI
# ==========================================
hasil = []


def evaluate_model(name, model):

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(
        mean_squared_error(y_test, pred)
    )

    hasil.append({
        'Model': name,
        'R2': r2,
        'MAE': mae,
        'RMSE': rmse
    })

    print("=" * 50)
    print(name)
    print("R2   :", r2)
    print("MAE  :", mae)
    print("RMSE :", rmse)


# ==========================================
# LINEAR REGRESSION
# ==========================================
evaluate_model(
    'Linear Regression',
    LinearRegression()
)


# ==========================================
# POLYNOMIAL DEGREE 2
# ==========================================
poly2 = Pipeline([
    (
        'poly',
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        'lr',
        LinearRegression()
    )
])

evaluate_model(
    'Polynomial Degree 2',
    poly2
)


# ==========================================
# POLYNOMIAL DEGREE 3
# ==========================================
poly3 = Pipeline([
    (
        'poly',
        PolynomialFeatures(
            degree=3,
            include_bias=False
        )
    ),
    (
        'lr',
        LinearRegression()
    )
])

evaluate_model(
    'Polynomial Degree 3',
    poly3
)


# ==========================================
# RANDOM FOREST
# ==========================================
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

evaluate_model(
    'Random Forest',
    rf
)


# ==========================================
# XGBOOST
# ==========================================
if xgb_available:

    xgb = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    evaluate_model(
        'XGBoost',
        xgb
    )


# ==========================================
# HASIL AKHIR
# ==========================================
hasil_df = pd.DataFrame(hasil)

hasil_df = hasil_df.sort_values(
    by='R2',
    ascending=False
)

print("\n")
print("=" * 60)
print("HASIL AKHIR")
print("=" * 60)
print(hasil_df)

FileNotFoundError: [Errno 2] No such file or directory: 'Data_Model_IoTMLCQ_2024(11).xlsx'